In [5]:
import qrisp
import src.classical.ec_arithmetic as clECarithm
import src.quantum.ec_arithmetic as qECarithm

In [ ]:
p=7
a=5
b=4
curve = clECarithm.EllCurve(a, b, p)
# Sample input points
input_points = [(2,1), (4,5), (0,2), (5,0), (0,5), (4,2), (2,6) ] 
G_ell = clECarithm.EllPoint(3, 5)
G_qrisp = [3, 5]
mod_p = qrisp.QuantumModulus(p)

# Compare outputs
for point in input_points:
    P_ell = clECarithm.EllPoint(point[0], point[1])
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = [point[0], point[1]]
    result_ell = clECarithm.ell_add_generic(P_ell, G_ell, curve)
    result_qrisp = qECarithm.qrisp_ell_add_inpl(anc, G_qrisp, p)    
    print("Input Point:", point)
    print("Elliptic addition:", (result_ell.x, result_ell.y))
    print(qrisp.multi_measurement(result_qrisp))
    print()

In [6]:
import cProfile
with cProfile.Profile() as pr:
    p=7
    a=5
    b=4

    curve = clECarithm.EllCurve(a, b, p)
    x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
    x1[:] = 1
    mod_p = qrisp.QuantumModulus(p)
    #store registers
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    #initialize the store register to the point P_0 = (3,2)
    anc[:] = [2,6]

    #Generator G = [2,2]
    G = [0,5]

    res = qECarithm.qrisp_ell_mult_add(G,anc,x1,curve)
    print(len([qb for qb in res[0].qs.qubits if qb.allocated]))
    res_compiled = res[0].qs.compile()
    print(res_compiled.depth())
    print(res_compiled.transpile().count_ops())
    #print(qrisp.multi_measurement(res))

9
121447
{'x': 2896, 'cx': 153384, 'h': 31338, 'p': 166428}


In [7]:
import pstats
results = pstats.Stats(pr).sort_stats(pstats.SortKey.TIME)
results.print_stats()

         305335722 function calls (304695777 primitive calls) in 256.790 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2  151.905   75.952  151.905   75.952 /home/dpolimen/Qrisp/src/qrisp/core/compilation.py:874(ancestors_jitted_wrapper)
      129   14.589    0.113   25.972    0.201 /home/dpolimen/Qrisp/src/qrisp/uncomputation/unqomp.py:75(dag_from_qc)
143490/12605   13.876    0.000   15.811    0.001 /home/dpolimen/Qrisp/src/qrisp/circuit/transpiler.py:90(transpile_inner)
       26    6.318    0.243  184.843    7.109 /home/dpolimen/Qrisp/src/qrisp/core/compilation.py:539(topological_sort)
  2239937    5.526    0.000    7.105    0.000 /home/dpolimen/quantum-elliptic-curve-logarithm/venv/lib/python3.9/site-packages/networkx/classes/digraph.py:643(add_edge)
 26661045    5.415    0.000    5.415    0.000 /home/dpolimen/Qrisp/src/qrisp/core/compilation.py:484(<lambda>)
 48569283    4.578    0.000    4.578    0.000 /home

In [ ]:
print(len([qb for qb in res[0].qs.qubits if qb.allocated]))
res_compiled = res[0].qs.compile()
print(res_compiled.depth())
print(res_compiled.transpile().count_ops())

In [8]:
print(qrisp.multi_measurement(res))

Simulating 37 qubits.. |███████████████████▌                                 | [ 37%]

SystemError: CPUDispatcher(<function coo_sparse_matrix_mult_jitted at 0x7f2902fa70d0>) returned a result with an error set

Simulating 37 qubits.. |███████████████████▌                                 | [ 37%]